# 📡 Transceiver & Hardware Governance Lab (Section II-C)

This notebook implements the **Quantitative 'Research Loop'** for **Section II-C (Transceiver and Hardware Abstractions)**.
It combines **Regex Mining** (for coverage) with **Groq LLM Verification** (for semantic precision) to enforce strict governance on hardware taxonomy (e.g., distinguishing *Coherent* vs *IM/DD*, *OPA* vs *RIS*).

## Research Objectives (Section II-C)
1. **Detection**: Identify hardware components (Lasers, Modulators, Detectors, Beamformers).
2. **Governance Check**: Validate canonical taxonomy (Canonical Tags, Symbol Conventions).
3. **Evidence Extraction**: Extract precise quotes and locators for the manuscript.

## Target Scope (II-C)
- **Sources**: Laser Diodes (LD), VCSEL, LED, Optical Freq Combs.
- **Modulation**: IM/DD, Coherent (MZM, IQ), OFDM variations (DCO, ACO).
- **Detection**: PIN/APD, Balanced PD, Coherent Receiver (Heterodyne/Homodyne).
- **Beamforming**: OPA (Optical Phased Arrays), RIS/Metasurfaces.
- **Impairments**: Linewidth, Phase Noise, Shot/Thermal Noise, Nonlinearity.


In [1]:
# @title 1. Install & Setup
!pip install -q groq

from google.colab import drive
import os
import glob
import json
import csv
import re
from collections import Counter
from groq import Groq
from google.colab import userdata

# 1.1 Mount Drive & Set Base Dir
drive.mount('/content/drive')
BASE_DIR = "/content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST"

if os.path.exists(BASE_DIR):
    os.chdir(BASE_DIR)
    print(f"✅ Working Directory set to: {os.getcwd()}")
else:
    print(f"❌ Path not found: {BASE_DIR}. Please check your Drive structure.")

# 1.2 Load API Key
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
    print("🔑 Groq API Key loaded.")
except Exception as e:
    print(f"⚠️ Error: {e}. Ensure 'GROQ_API_KEY' is in Colab Secrets.")

# 1.3 Define Output Path
OUTPUT_DIR = "analysis/II_ev_v2"
os.makedirs(OUTPUT_DIR, exist_ok=True)
EVIDENCE_CSV = os.path.join(OUTPUT_DIR, "section2C_evidence.csv")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 5.5 MB/s eta 0:00:00
Mounted at /content/drive
✅ Working Directory set to: /content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST
🔑 Groq API Key loaded.


In [2]:
# @title 2. Data Loader (Recursive)
def load_processed_markdowns(target_ids=None, limit=None):
    """
    Loads markdown files recursively from 'data/proc_markdowns'.
    Matches the logic of previous Governance Labs.
    """
    search_path = os.path.join("data", "processed_markdowns")

    # Recursive search for all .md files
    all_files = glob.glob(os.path.join(search_path, "**", "*.md"), recursive=True)

    # Filter for O_ISAC or COMST files
    valid_files = [f for f in all_files if "O_ISAC" in f or "COMST" in f]

    # ID Filtering
    selected_files = []
    if target_ids:
        print(f"Applying filter for {len(target_ids)} Target IDs...")
        for f_path in valid_files:
            p_id = os.path.basename(f_path).replace('.md', '')
            if p_id in target_ids:
                selected_files.append(f_path)
    else:
        selected_files = valid_files

    if limit:
        selected_files = selected_files[:limit]

    print(f"Found {len(valid_files)} total files. Loading {len(selected_files)} for analysis.")

    data = []
    for f_path in selected_files:
        p_id = os.path.basename(f_path).replace('.md', '')
        try:
            with open(f_path, 'r', encoding='utf-8') as f:
                content = f.read()
                data.append((p_id, content))
        except Exception as e:
            print(f"Error reading {f_path}: {e}")

    return data

In [3]:
# @title 3. Define II-C Research Agent (Hardware Extraction)
def analyze_hardware_governance(paper_text, paper_id):
    """
    Asks LLM to extract Transceiver/Hardware details from the paper.
    Includes 'Reasoning' for Evidence Locking.
    """

    system_prompt = """
    You are a Senior Hardware Architect auditing Optical Wireless Communication papers for Section II-C (Transceivers).
    Your goal is to extract EXACT evidence of usage for:
    1. **Sources**: Laser Diodes, VCSELs, LEDs, Frequency Combs.
    2. **Modulation**: IM/DD vs Coherent (MZM, IQ Modulators).
    3. **Detectors**: PIN, APD, Balanced PD, Coherent Receivers.
    4. **Beamforming**: OPA, RIS, Lens Arrays.
    5. **Impairments**: Linewidth, Phase Noise, Nonlinearity.

    # CRITICAL RULES:
    - Distinguish 'Experimental Setup' vs 'Simulation Model'.
    - Capture specific parameters if mentioned (e.g., 'linewidth of 100 kHz').
    - IGNORE Related Work mentions.

    # OUTPUT FORMAT:
    Return a JSON object with a 'reasoning' string and a 'findings' list.
    Example:
    {
      "reasoning": "The authors describe an experimental setup in Section III using a DFB laser and an Avalon APD. They also simulate a Coherent system in Section IV.",
      "findings": [
         { "category": "Source", "tag": "DFB Laser", "status": "USED (Experiment)", "evidence": "we utilize a DFB laser at 1550nm..." },
         { "category": "Detector", "tag": "APD", "status": "USED (Experiment)", "evidence": "receiver employs a high-sensitivity APD..." }
      ]
    }
    """

    user_prompt = f"""
    Paper ID: {paper_id}

    Analyze this text strictly. DO NOT hallucinate.

    Text Content (First 35k chars):
    {paper_text[:35000]}
    """

    try:
        completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            model="llama-3.3-70b-versatile",
            response_format={"type": "json_object"},
            temperature=0
        )
        return json.loads(completion.choices[0].message.content)
    except Exception as e:
        return {"error": str(e), "reasoning": "Fail", "findings": []}


In [5]:
# @title 4. Execute Research Loop (Targeted)

# ==========================================
# CONFIGURATION
# ==========================================
# Recommended Papers for Hardware Diversity (from previous scans)
# O_ISAC_029 (Hybrid), O_ISAC_001 (IM/DD), O_ISAC_061 (FSO), O_ISAC_199 (VLC)
TARGET_PAPERS = None
LIMIT = None
OUTPUT_CSV = "analysis/II_ev_v2/section2C_evidence_LLM.csv"
# ==========================================

# 1. Load Papers
papers = load_processed_markdowns(target_ids=TARGET_PAPERS, limit=LIMIT)

# 2. Run Agent
all_findings = []
print(f"\n🚀 Starting II-C Hardware Analysis on {len(papers)} papers...\n")

for pid, text in papers:
    print(f"Processing {pid}...")
    result = analyze_hardware_governance(text, pid)

    reasoning = result.get("reasoning", "No reasoning provided.")
    findings = result.get("findings", [])

    print(f"  🧠 Agent Reasoning: {reasoning[:150]}...")

    for f in findings:
        f["paper_id"] = pid
        f["full_reasoning"] = reasoning
        all_findings.append(f)
        print(f"     -> 🛠️ Found {f.get('category')}: {f.get('tag')}")
    print("-"*40)

# 3. Export Results
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
if all_findings:
    keys = ["paper_id", "category", "tag", "status", "evidence", "full_reasoning"]
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(all_findings)
    print(f"\n✅ Saved {len(all_findings)} confirmed evidence rows to {OUTPUT_CSV}")
    # Show first few results
    import pandas as pd
    display(pd.read_csv(OUTPUT_CSV).head())
else:
    print("\n⚠️ No findings extracted.")

Found 313 total files. Loading 313 for analysis.

🚀 Starting II-C Hardware Analysis on 313 papers...

Processing O_ISAC_029...
  🧠 Agent Reasoning: The authors describe an experimental setup in Section III using a thin-film lithium niobate Mach-Zehnder modulator (TFLN-MZM) for direct reception and...
     -> 🛠️ Found Source: External Cavity Laser (ECL)
     -> 🛠️ Found Modulation: Coherent (MZM, IQ Modulators)
     -> 🛠️ Found Detector: Balanced PD
     -> 🛠️ Found Beamforming: None
     -> 🛠️ Found Impairments: Frequency Instability
----------------------------------------
Processing O_ISAC_029...
  🧠 Agent Reasoning: The authors describe an experimental setup in Section III using a thin-film lithium niobate Mach-Zehnder modulator (TFLN-MZM) for direct reception and...
     -> 🛠️ Found Modulation: Direct LFM Reception and De-chirping
     -> 🛠️ Found Source: External Cavity Laser
     -> 🛠️ Found Detector: Photodiode (PD) and Balanced PD (BPD)
     -> 🛠️ Found Modulation: IQ Modulatio

,paper_id,category,tag,status,evidence,full_reasoning
0,O_ISAC_029,Source,External Cavity Laser (ECL),USED (Experiment),The ISAC signal is generated in the CU/DU and ...,The authors describe an experimental setup in ...
1,O_ISAC_029,Modulation,"Coherent (MZM, IQ Modulators)",USED (Experiment),The IQM consists of two MZMs and an optical ph...,The authors describe an experimental setup in ...
2,O_ISAC_029,Detector,Balanced PD,USED (Experiment),By combining this approach with coherent homod...,The authors describe an experimental setup in ...
3,O_ISAC_029,Beamforming,NaN,NOT USED,No beamforming techniques are mentioned in the...,The authors describe an experimental setup in ...
4,O_ISAC_029,Impairments,Frequency Instability,MITIGATED,The use of TFLN-MZM for PDC eliminates the fre...,The authors describe an experimental setup in ...
